In [1]:
# Generate the data to be tested.
import torch
import pandas as pd

# Configuration
num_wings = 10
num_points = 9000

# 1. Generate X coordinates (10 wings, 9000 points each)
x_data = torch.linspace(0, 1, num_points).repeat(num_wings, 1)
x_columns = [f"x_{i}" for i in range(num_wings)]
df_x = pd.DataFrame(x_data.numpy().T, columns=x_columns)

# 2. Generate Y coordinates (10 wings, 100 points each)
y_data = torch.rand(num_wings, num_points)
y_columns = [f"y_{i}" for i in range(num_wings)]
df_y = pd.DataFrame(y_data.numpy().T, columns=y_columns)

# 3. Generate Alpha values with specific aerodynamic titles
# Columns represent: Lift (Cl), Drag (Cd), Pitching Moment (Cm), and Center of Pressure (Cp)
alpha_columns = ["Cl", "Cd", "Cm", "Cp"]

# Alpha values for 4 degrees
alpha_4_data = torch.randn(num_points, 4)
df_alpha_4 = pd.DataFrame(alpha_4_data.numpy(), columns=alpha_columns)

# Alpha values for 15 degrees
alpha_15_data = torch.randn(num_points, 4)
df_alpha_15 = pd.DataFrame(alpha_15_data.numpy(), columns=alpha_columns)

# Saving the CSV files
df_x.to_csv("dataset/5-x_coords.csv", index=True)
df_y.to_csv("dataset/5-y_coords.csv", index=True)
df_alpha_4.to_csv("dataset/5-alpha_4.csv", index=True)
df_alpha_15.to_csv("dataset/5-alpha_15.csv", index=True)

In [2]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

In [3]:
# Read Data
X = pd.read_csv("dataset/5-x_coords.csv")
Y = pd.read_csv("dataset/5-y_coords.csv")
A4 = pd.read_csv("dataset/5-alpha_4.csv")
A15 = pd.read_csv("dataset/5-alpha_15.csv")
XY = X.merge(Y)

print(X.shape)
print(Y.shape)
print(A4.shape)
print(A15.shape)
print(XY.shape)
XY

(9000, 11)
(9000, 11)
(9000, 5)
(9000, 5)
(9000, 21)


,Unnamed: 0,x_0,x_1,x_2,x_3,x_4,x_5,x_6,x_7,x_8,...,y_0,y_1,y_2,y_3,y_4,y_5,y_6,y_7,y_8,y_9
0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.912759,0.860602,0.928463,0.704914,0.210809,0.655263,0.392768,0.009377,0.847961,0.468246
1,1,0.000111,0.000111,0.000111,0.000111,0.000111,0.000111,0.000111,0.000111,0.000111,...,0.875839,0.103062,0.170802,0.956622,0.619357,0.700129,0.557622,0.276160,0.112219,0.960549
2,2,0.000222,0.000222,0.000222,0.000222,0.000222,0.000222,0.000222,0.000222,0.000222,...,0.984665,0.254467,0.779247,0.484557,0.564613,0.807746,0.754795,0.315535,0.871422,0.188619
3,3,0.000333,0.000333,0.000333,0.000333,0.000333,0.000333,0.000333,0.000333,0.000333,...,0.275432,0.769255,0.747117,0.897832,0.029427,0.820235,0.568323,0.549203,0.729423,0.485141
4,4,0.000444,0.000444,0.000444,0.000444,0.000444,0.000444,0.000444,0.000444,0.000444,...,0.970081,0.216735,0.161981,0.163824,0.124586,0.676254,0.611214,0.920016,0.882556,0.593772
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8995,8995,0.999556,0.999556,0.999556,0.999556,0.999556,0.999556,0.999556,0.999556,0.999556,...,0.675533,0.311156,0.015039,0.949019,0.764040,0.804182,0.435023,0.646854,0.484986,0.890854
8996,8996,0.999667,0.999667,0.999667,0.999667,0.999667,0.999667,0.999667,0.999667,0.999667,...,0.899807,0.601061,0.173323,0.064273,0.819877,0.278545,0.622947,0.739965,0.728123,0.899580
8997,8997,0.999778,0.999778,0.999778,0.999778,0.999778,0.999778,0.999778,0.999778,0.999778,...,0.561210,0.623094,0.236658,0.359674,0.891124,0.313929,0.827788,0.279019,0.293194,0.579578
8998,8998,0.999889,0.999889,0.999889,0.999889,0.999889,0.999889,0.999889,0.999889,0.999889,...,0.089987,0.931385,0.460239,0.238366,0.161689,0.932912,0.306680,0.316399,0.247759,0.111968


In [4]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.f1 = nn.Linear(20, 16)
        self.f2 = nn.ReLU()
        self.f3 = nn.Linear(16, 4) 

    def forward(self, x):
        x = self.f1(x)
        x = self.f2(x)
        return self.f3(x)
    
class Data(torch.utils.data.Dataset):
    def __init__(self, features, targets):
        self.features = torch.tensor(features.values, dtype = torch.float32)
        self.targets = torch.tensor(targets.values, dtype = torch.float32)
    def __len__(self):
        return self.features.shape[0]
    
    def __getitem__(self, idx):
        return self.features[idx], self.targets[idx]


In [5]:
dataset4 = XY.merge(A4)
data4_df = dataset4[['x_0', 'x_1', 'x_2', 'x_3', 'x_4', 'x_5', 'x_6', 'x_7',
       'x_8', 'x_9', 'y_0', 'y_1', 'y_2', 'y_3', 'y_4', 'y_5', 'y_6', 'y_7',
       'y_8', 'y_9', 'Cl', 'Cd', 'Cm', 'Cp']]
data4_ds = Data(data4_df[['x_0', 'x_1', 'x_2', 'x_3', 'x_4', 'x_5', 'x_6', 'x_7',
       'x_8', 'x_9', 'y_0', 'y_1', 'y_2', 'y_3', 'y_4', 'y_5', 'y_6', 'y_7',
       'y_8', 'y_9']], data4_df[['Cl', 'Cd', 'Cm', 'Cp']])
n = data4_df.shape[0]
n_train = int(0.9 * n)
n_val = n - n_train

train_dataset, val_dataset = torch.utils.data.random_split(data4_ds, [n_train, n_val])
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = 32, shuffle = True)
data_loader = DataLoader(data4_df, batch_size = 32, shuffle = True)

model = Model()
epochs = 5
loss_fn = torch.nn.MSELoss()
optim = torch.optim.Adam(model.parameters(), lr = 0.01)

In [6]:
# Train the model

for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    batch_loss = 0.0
    n_batch = 0
    for x_train, y_train in train_loader:
        output = model(x_train)
        optim.zero_grad()
        loss = loss_fn(output, y_train)
        loss.backward()
        optim.step()
        batch_loss += loss.item()
        n_batch += 1
    print(f"Epoch {epoch} Error {batch_loss / n_batch:.6f}")

# Evaluate the Model:

model.eval()
n_batch = 0
batch_loss = 0.0
with torch.no_grad():
    for x_val, y_val in val_loader:
        output = model(x_val)
        loss = loss_fn(output, y_val)
        batch_loss += loss.item()
        n_batch += 1
print(f"Validation Error: {batch_loss / n_batch:0.6f}")
        

Epoch 0 Error 1.020304
Epoch 1 Error 1.017904
Epoch 2 Error 1.017669
Epoch 3 Error 1.017313
Epoch 4 Error 1.017854
Validation Error: 0.993490
